# Демонстрация покупки БАС через систему Эксплуатант

Этот notebook демонстрирует новую функциональность системы Эксплуатант для покупки БАС у Разработчиков.

## Изменения согласно Change Request #1

1. Интеграция с Регулятором для получения списка зарегистрированных систем
2. Получение каталогов БАС от Разработчиков
3. Покупка БАС через Fleet Manager
4. Автоматическое добавление купленных БАС в парк

In [1]:
# Импорт необходимых библиотек
import sys
import os
import asyncio
import yaml
from datetime import datetime
from unittest.mock import Mock, AsyncMock

# Добавляем путь к корню проекта (репозитория)
sys.path.append(os.path.abspath("../../../"))

# Базовые настройки окружения для демонстрации
os.environ.setdefault("SYSTEM_ID", "operator-demo-001")
os.environ.setdefault("API_VERSION", "v1")

# Импорт компонентов системы
from systems.operator.src.regulator_client import RegulatorClient
from systems.operator.src.developer_client import DeveloperClient, UASCategory
from systems.operator.src.fleet_manager import FleetManager
from systems.operator.src.topics import FleetManagerActions, SystemTopics, ComponentTopics

## Диаграммы сценариев (PlantUML)

Ниже приведены диаграммы последовательности для ключевых сценариев демонстрации. Код диаграмм хранится прямо в блокноте, чтобы сценарий был самодокументируемым.

### Сценарий A: Покупка БАС через Эксплуатанта

```plantuml
@startuml
title Покупка БАС через Эксплуатанта

actor "Эксплуатант" as OP
participant "Fleet Manager" as FM
participant "RegulatorClient" as RC
participant "DeveloperClient" as DC
participant "Разработчик" as DEV

OP -> RC: get_system_topics()
RC --> OP: topics (developer topics)

OP -> DC: get_all_catalogs()
DC -> DEV: get_catalog
DEV --> DC: catalog
DC --> OP: catalogs

OP -> FM: purchase_uas(developer_id, model_id, quantity)
FM -> DC: purchase_uas(...)
DC -> DEV: purchase_uas(...)
DEV --> DC: purchase_result
DC --> FM: purchase_result
FM --> OP: success + добавление БАС в парк

@enduml
```

### Сценарий B: Контроль безопасности при межкомпонентном запросе

```plantuml
@startuml
title Валидация запроса через Security Monitor

participant "Operator System" as OS
participant "Security Monitor" as SM
participant "Fleet Manager" as FM

OS -> SM: validate_request(action, sender, context)
SM --> OS: allowed/denied

alt allowed
  OS -> FM: find_available_uas(requirements)
  FM --> OS: list of suitable UAS
else denied
  OS --> OS: отказ обработки / запись нарушения
end

@enduml
```

## 1. Настройка окружения и создание mock компонентов

In [2]:
# Создаём mock для SystemBus
mock_bus = Mock()
mock_bus.request = AsyncMock()
mock_bus.publish = AsyncMock()

# Настраиваем переменные окружения для тестового режима
os.environ['AGGREGATOR_ID'] = 'agg-demo-001'
os.environ['DEVELOPERS_IDS'] = 'aeronext-001,skytech-002,droneworks-003'
os.environ['INSURANCE_IDS'] = 'ins-demo-001'
os.environ['UTM_ID'] = 'utm-demo-001'

print("✅ Окружение настроено")
print(f"   - Агрегатор: {os.environ['AGGREGATOR_ID']}")
print(f"   - Разработчики: {os.environ['DEVELOPERS_IDS']}")
print(f"   - Страховая: {os.environ['INSURANCE_IDS']}")
print(f"   - ОрВД: {os.environ['UTM_ID']}")

✅ Окружение настроено
   - Агрегатор: agg-demo-001
   - Разработчики: aeronext-001,skytech-002,droneworks-003
   - Страховая: ins-demo-001
   - ОрВД: utm-demo-001


## 2. Создание клиентов для взаимодействия с внешними системами

In [3]:
# Создаём клиент для Регулятора
regulator_client = RegulatorClient(mock_bus)

# Получаем список зарегистрированных систем
topics = await regulator_client.get_system_topics()

print("📋 Зарегистрированные системы:")
for key, topic_info in topics.items():
    print(f"   - {topic_info.system_type}: {topic_info.system_id} (topic: {topic_info.topic})")

📋 Зарегистрированные системы:
   - aggregator: agg-demo-001 (topic: systems.aggregator.agg-demo-001)
   - developer: aeronext-001 (topic: systems.developer.aeronext-001)
   - developer: skytech-002 (topic: systems.developer.skytech-002)
   - developer: droneworks-003 (topic: systems.developer.droneworks-003)
   - insurer: ins-demo-001 (topic: systems.insurer.ins-demo-001)
   - utm: utm-demo-001 (topic: systems.utm.utm-demo-001)


In [4]:
# Создаём клиент для Разработчиков
developer_client = DeveloperClient(mock_bus, regulator_client)

# Используем YAML файл с каталогом (тестовые данные в resources)
developer_client.yaml_catalog_path = "operator_clients/resources/developers_catalog.yaml"

print("✅ Клиенты созданы")

✅ Клиенты созданы


## 3. Получение каталогов БАС от Разработчиков

In [5]:
# Получаем все каталоги
catalogs = await developer_client.get_all_catalogs()

print(f"📚 Получено каталогов: {len(catalogs)}\n")

# Выводим информацию о каждом разработчике
for dev_id, catalog in catalogs.items():
    print(f"🏢 {catalog.developer_name} ({dev_id})")
    print(f"   Контакты: {catalog.contact_info.get('email', 'N/A')}")
    print(f"   Моделей в каталоге: {len(catalog.models)}")
    
    for model in catalog.models[:2]:  # Показываем первые 2 модели
        print(f"\n   🚁 {model.name} ({model.model_id})")
        print(f"      - Категория: {model.category}")
        print(f"      - Грузоподъёмность: {model.specifications.get('max_payload_kg', 0)} кг")
        print(f"      - Дальность: {model.specifications.get('max_range_km', 0)} км")
        print(f"      - Цена: {model.price:,.0f} ₽")
        print(f"      - Доступно: {model.available_quantity} шт.")
    
    if len(catalog.models) > 2:
        print(f"\n   ... и ещё {len(catalog.models) - 2} моделей")
    print("\n" + "="*60 + "\n")

📚 Получено каталогов: 2

🏢 AeroTech Solutions (dev-001)
   Контакты: sales@aerotech.com
   Моделей в каталоге: 2

   🚁 CargoLite 100 (AT-LC100)
      - Категория: UASCategory.LIGHT_CARGO
      - Грузоподъёмность: 5.0 кг
      - Дальность: 50.0 км
      - Цена: 150,000 ₽
      - Доступно: 10 шт.

   🚁 CargoMax 200 (AT-HC200)
      - Категория: UASCategory.HEAVY_CARGO
      - Грузоподъёмность: 20.0 кг
      - Дальность: 30.0 км
      - Цена: 350,000 ₽
      - Доступно: 5 шт.


🏢 DroneWorks Industries (dev-002)
   Контакты: orders@droneworks.ru
   Моделей в каталоге: 2

   🚁 AgroSpray 300 (DW-AG300)
      - Категория: UASCategory.AGRO
      - Грузоподъёмность: 15.0 кг
      - Дальность: 20.0 км
      - Цена: 280,000 ₽
      - Доступно: 8 шт.

   🚁 Inspector Pro (DW-IN400)
      - Категория: UASCategory.INSPECTOR
      - Грузоподъёмность: 2.0 кг
      - Дальность: 100.0 км
      - Цена: 200,000 ₽
      - Доступно: 15 шт.




## 4. Поиск подходящих БАС по требованиям

In [6]:
# Определяем требования для доставки
requirements = {
    'category': UASCategory.LIGHT_CARGO,
    'min_payload': 3.0,  # минимум 3 кг
    'min_range': 30.0,   # минимум 30 км
    'require_certification': True,
    'required_safety_features': ['Parachute recovery system']
}

print("🔍 Требования к БАС:")
for key, value in requirements.items():
    print(f"   - {key}: {value}")

# Ищем подходящие модели
suitable_models = developer_client.find_best_uas_for_requirements(requirements)

print(f"\n✅ Найдено подходящих моделей: {len(suitable_models)}\n")

# Выводим информацию о подходящих моделях
for i, model in enumerate(suitable_models[:5], 1):
    print(f"{i}. {model.name} ({model.model_id})")
    print(f"   - Разработчик: {model.manufacturer}")
    print(f"   - Грузоподъёмность: {model.specifications.get('max_payload_kg', 0)} кг")
    print(f"   - Дальность: {model.specifications.get('max_range_km', 0)} км")
    print(f"   - Цена: {model.price:,.0f} ₽")
    print(f"   - Функции безопасности: {', '.join(model.safety_features[:2])}...")
    print()

🔍 Требования к БАС:
   - category: UASCategory.LIGHT_CARGO
   - min_payload: 3.0
   - min_range: 30.0
   - require_certification: True
   - required_safety_features: ['Parachute recovery system']

✅ Найдено подходящих моделей: 1

1. CargoLite 100 (AT-LC100)
   - Разработчик: AeroTech Solutions
   - Грузоподъёмность: 5.0 кг
   - Дальность: 50.0 км
   - Цена: 150,000 ₽
   - Функции безопасности: Redundant flight controller, Parachute recovery system...



## 5. Создание Fleet Manager и покупка БАС

In [7]:
# Создаём Fleet Manager (передаём клиентов через config)
fleet_manager = FleetManager(
    "fleet-demo",
    mock_bus,
    config={
        "developer_client": developer_client,
        "regulator_client": regulator_client,
    },
)

# Проверяем текущий парк
initial_fleet = fleet_manager._handle_get_uas_list({})
print(f"📊 Текущий парк БАС: {initial_fleet.get('total_count', initial_fleet.get('total', 0))} единиц\n")

# Выбираем первую подходящую модель для покупки
if suitable_models:
    selected_model = suitable_models[0]
    developer_id = None

    # Находим разработчика этой модели
    for dev_id, catalog in catalogs.items():
        for model in catalog.models:
            if model.model_id == selected_model.model_id:
                developer_id = dev_id
                break
        if developer_id:
            break

    print("🛒 Покупаем БАС:")
    print(f"   - Модель: {selected_model.name}")
    print(f"   - Разработчик: {developer_id}")
    print("   - Количество: 2 шт.")
    print(f"   - Цена за единицу: {selected_model.price:,.0f} ₽")

📊 Текущий парк БАС: 0 единиц

🛒 Покупаем БАС:
   - Модель: CargoLite 100
   - Разработчик: dev-001
   - Количество: 2 шт.
   - Цена за единицу: 150,000 ₽


In [8]:
# Выполняем покупку
purchase_request = {
    'action': FleetManagerActions.PURCHASE_UAS,
    'payload': {
        'developer_id': developer_id,
        'model_id': selected_model.model_id,
        'quantity': 2
    }
}

purchase_result = fleet_manager._handle_purchase_uas(purchase_request)

if purchase_result['success']:
    print("\n✅ Покупка успешна!")
    print(f"   - ID заказа: {purchase_result['order_id']}")
    print(f"   - Общая стоимость: {purchase_result['total_price']:,.0f} ₽")
    print(f"   - Срок поставки: {purchase_result['delivery_time_days']} дней")
else:
    print(f"\n❌ Ошибка покупки: {purchase_result.get('error', 'Unknown error')}")


✅ Покупка успешна!
   - ID заказа: PO-20260317143220
   - Общая стоимость: 300,000 ₽
   - Срок поставки: 14 дней


## 6. Проверка обновлённого парка БАС

In [9]:
# Получаем обновлённый список БАС
updated_fleet = fleet_manager._handle_get_uas_list({})

print(f"📊 Обновлённый парк БАС: {updated_fleet['total_count']} единиц\n")
print("Список БАС в парке:")

for i, uas in enumerate(updated_fleet['uas_list'], 1):
    print(f"\n{i}. БАС {uas['id']}")
    print(f"   - Модель: {uas['model_id']}")
    print(f"   - Тип: {uas['type']}")
    print(f"   - Статус: {uas['status']}")
    print(f"   - Заряд батареи: {uas['battery_level']*100:.0f}%")
    print(f"   - Грузоподъёмность: {uas['max_payload']} кг")
    print(f"   - Дальность: {uas['max_range']} км")

📊 Обновлённый парк БАС: 2 единиц

Список БАС в парке:

1. БАС UAS-001
   - Модель: AT-LC100
   - Тип: light_cargo
   - Статус: available
   - Заряд батареи: 100%
   - Грузоподъёмность: 5.0 кг
   - Дальность: 50.0 км

2. БАС UAS-002
   - Модель: AT-LC100
   - Тип: light_cargo
   - Статус: available
   - Заряд батареи: 100%
   - Грузоподъёмность: 5.0 кг
   - Дальность: 50.0 км


## 7. Демонстрация поиска доступных БАС для миссии

In [10]:
# Ищем доступные БАС для конкретной миссии
mission_requirements = {
    'min_payload': 3.5,
    'min_range': 35.0,
    'min_battery': 0.9
}

find_request = {
    'action': FleetManagerActions.FIND_AVAILABLE_UAS,
    'payload': {
        'requirements': mission_requirements
    }
}

available_uas = fleet_manager._handle_find_available_uas(find_request)

print("🔍 Поиск БАС для миссии:")
print(f"   Требования: груз {mission_requirements['min_payload']} кг, ")
print(f"               дальность {mission_requirements['min_range']} км, ")
print(f"               заряд ≥ {mission_requirements['min_battery']*100:.0f}%")
print(f"\n✅ Найдено подходящих БАС: {available_uas['count']}")

for uas in available_uas['suitable_uas']:
    print(f"\n   - {uas['id']} ({uas['model_id']})")
    print(f"     Заряд: {uas['battery_level']*100:.0f}%, ")
    print(f"     Груз: до {uas['max_payload']} кг, ")
    print(f"     Дальность: до {uas['max_range']} км")

🔍 Поиск БАС для миссии:
   Требования: груз 3.5 кг, 
               дальность 35.0 км, 
               заряд ≥ 90%

✅ Найдено подходящих БАС: 2

   - UAS-001 (AT-LC100)
     Заряд: 100%, 
     Груз: до 5.0 кг, 
     Дальность: до 50.0 км

   - UAS-002 (AT-LC100)
     Заряд: 100%, 
     Груз: до 5.0 кг, 
     Дальность: до 50.0 км


## 8. Резервирование БАС для миссии

In [11]:
if available_uas['count'] > 0:
    # Резервируем первый доступный БАС
    selected_uas = available_uas['suitable_uas'][0]
    
    reserve_request = {
        'action': FleetManagerActions.RESERVE_UAS,
        'payload': {
            'uas_id': selected_uas['id'],
            'mission_id': 'MISSION-DEMO-001',
            'duration': 7200  # 2 часа
        }
    }
    
    reserve_result = fleet_manager._handle_reserve_uas(reserve_request)
    
    if reserve_result['reserved']:
        print(f"✅ БАС {selected_uas['id']} зарезервирован для миссии MISSION-DEMO-001")
        print(f"   Резервация действует до: {reserve_result['expires_at']}")
    else:
        print("❌ Не удалось зарезервировать БАС")
else:
    print("❌ Нет доступных БАС для резервирования")

✅ БАС UAS-001 зарезервирован для миссии MISSION-DEMO-001
   Резервация действует до: None


## Заключение

В этой демонстрации мы показали:

1. **Интеграцию с Регулятором** - получение списка зарегистрированных систем
2. **Работу с каталогами Разработчиков** - получение и анализ доступных моделей БАС
3. **Поиск подходящих БАС** - фильтрация по требованиям миссии
4. **Покупку БАС** - оформление заказа и добавление в парк
5. **Управление парком** - поиск доступных БАС и их резервирование

Эта функциональность позволяет Эксплуатанту:
- Автоматически расширять парк БАС по мере необходимости
- Выбирать оптимальные модели для конкретных задач
- Интегрироваться с экосистемой беспилотной авиации